In [10]:
"""基于 everyuser / k-means 接口的可视化 Web 原型

使用说明：
1. 确保当前工作目录下已存在并可运行 `everyuser.ipynb` 与 `k-means.ipynb`。
2. 先在它们各自的 notebook 中运行核心函数单元（或在本文件中通过 %run 自动加载）。
3. 在本 notebook 中继续编写绘图与 Web 展示相关代码。
"""

import pandas as pd
import numpy as np
import plotly.express as px

# 后续单元将：
# - 调用 everyuser.ipynb 中的接口生成用户画像数据
# - 使用 Plotly 画出交互式图表
# - 演示如何导出为独立 HTML 网页


In [11]:
# 1. 从 everyuser 逻辑加载并预处理数据（在本 notebook 中直接实现接口）

import logging

logging.basicConfig(level=logging.INFO)

# === 直接拷贝 everyuser.ipynb 中的数据接口函数，实现完全自包含 ===

def load_data(path="customer_clusters_simple.csv"):
    """从 CSV 加载用户特征数据并返回 DataFrame。抛出异常由调用方处理。"""
    df = pd.read_csv(path)
    return df


def create_individual_profiles(df: pd.DataFrame) -> pd.DataFrame:
    """将原始特征表标准化为统一的用户画像表格。返回新的 DataFrame。
    兼容多种列名，尽量使用 .get 以避免 KeyError。
    """
    profiles = []

    for _, row in df.iterrows():
        # 安全地读取数值：只在值为 None 或 NaN 时才使用默认值，不用 "or" 避免把 0 当成缺失
        def _safe_float(v, default=0.0):
            if v is None or (isinstance(v, float) and np.isnan(v)):
                return float(default)
            return float(v)

        def _safe_int(v, default=0):
            if v is None or (isinstance(v, float) and np.isnan(v)):
                return int(default)
            return int(v)

        total_sales = _safe_float(row.get("Total_Sales", 0.0), 0.0)
        total_orders = _safe_int(row.get("Total_Orders", 0), 0)
        avg_order_value = _safe_float(row.get("Avg_Order_Value", 0.0), 0.0)
        lifetime_val = row.get("Customer_Lifetime_Days", row.get("Customer_Lifetime", 0))
        customer_lifetime = _safe_float(lifetime_val, 0.0)
        purchase_freq = _safe_float(row.get("Purchase_Frequency", 0.0), 0.0)
        profit_margin = _safe_float(row.get("Profit_Margin", 0.0), 0.0)

        last_purchase_raw = row.get("Days_Since_Last_Purchase", row.get("Last_Purchase_Days_Ago", np.nan))
        # 对最近一次购买天数：NaN 才当缺失，0 是“刚购买”
        if last_purchase_raw is None or (isinstance(last_purchase_raw, float) and np.isnan(last_purchase_raw)):
            last_purchase_days_ago = np.nan
        else:
            last_purchase_days_ago = float(last_purchase_raw)

        engagement_raw = row.get("Total_Engagement_Score", row.get("Engagement_Score", 0.0))
        engagement_score = _safe_float(engagement_raw, 0.0)

        unique_products_raw = row.get("Unique_Products_Purchased", row.get("Unique_Products_Bought", 0))
        unique_products_bought = _safe_int(unique_products_raw, 0)

        profile = {
            "Customer_ID": row.get("Customer ID", row.get("Customer_ID", None)),
            "Customer_Name": row.get("Customer Name", row.get("Customer_Name", "Unknown")),
            "Cluster": row.get("Cluster", None),
            "Cluster_Label": row.get("Cluster_Label", None),
            "Total_Sales": total_sales,
            "Total_Orders": total_orders,
            "Avg_Order_Value": avg_order_value,
            "Customer_Lifetime": customer_lifetime,
            "Purchase_Frequency": purchase_freq,
            "Profit_Margin": profit_margin,
            "Last_Purchase_Days_Ago": last_purchase_days_ago,
            "Engagement_Score": engagement_score,
            "Segment": row.get("Segment", None),
            "Region": row.get("Region", None),
            "Country": row.get("Country", None),
            "Gender": row.get("Gender", None),
            "Age": row.get("Age", np.nan),
            "Education": row.get("Education", None),
            "Marital_Status": row.get("Marital Status", row.get("Marital_Status", None)),
            "Favorite_Product": row.get("Favorite_Product", "Unknown"),
            "Unique_Products_Bought": unique_products_bought,
        }
        profiles.append(profile)

    return pd.DataFrame(profiles)


def calculate_percentiles_and_rankings(df: pd.DataFrame, metrics=None) -> pd.DataFrame:
    """为指定指标计算百分位、排名、评级和与均值对比。返回新 DataFrame。"""
    result = df.copy()
    if metrics is None:
        metrics = [
            "Total_Sales",
            "Total_Orders",
            "Avg_Order_Value",
            "Purchase_Frequency",
            "Profit_Margin",
            "Engagement_Score",
            "Customer_Lifetime",
        ]

    def get_grade(percentile):
        if percentile >= 80:
            return "A"
        elif percentile >= 60:
            return "B"
        elif percentile >= 40:
            return "C"
        elif percentile >= 20:
            return "D"
        else:
            return "E"

    for metric in metrics:
        if metric in result.columns:
            result[f"{metric}_Percentile"] = result[metric].rank(pct=True) * 100
            result[f"{metric}_Rank"] = result[metric].rank(ascending=False, method="min")
            result[f"{metric}_Grade"] = result[f"{metric}_Percentile"].apply(get_grade)
            overall_mean = result[metric].mean()
            # 避免除以零
            if overall_mean != 0 and not np.isnan(overall_mean):
                result[f"{metric}_vs_Mean"] = ((result[metric] - overall_mean) / overall_mean * 100).round(1)
            else:
                result[f"{metric}_vs_Mean"] = 0.0

    return result


# === 使用上述接口生成画像数据 ===

csv_path = "customer_clusters_simple.csv"  # 如有需要可在这里修改为你的真实路径

raw_df = load_data(csv_path)
profiles_df = create_individual_profiles(raw_df)
profiles_scored_df = calculate_percentiles_and_rankings(profiles_df)

print("原始数据形状:", raw_df.shape)
print("画像数据形状:", profiles_scored_df.shape)

# 预览前几行
profiles_scored_df.head()

原始数据形状: (795, 60)
画像数据形状: (795, 48)


,Customer_ID,Customer_Name,Cluster,Total_Sales,Total_Orders,Avg_Order_Value,Customer_Lifetime,Purchase_Frequency,Profit_Margin,Last_Purchase_Days_Ago,...,Profit_Margin_Grade,Profit_Margin_vs_Mean,Engagement_Score_Percentile,Engagement_Score_Rank,Engagement_Score_Grade,Engagement_Score_vs_Mean,Customer_Lifetime_Percentile,Customer_Lifetime_Rank,Customer_Lifetime_Grade,Customer_Lifetime_vs_Mean
0,AB-00363,Mcintyre Yedwab,3,8960.0,58,154.48,1068.0,0.0543,0.4612,13.0,...,C,-0.6,63.710692,286.0,B,10.5,58.553459,321.0,C,0.8
1,AB-00421,Whitney Yedwab,1,7840.0,45,174.22,1042.0,0.0432,0.4905,34.0,...,A,5.7,2.767296,773.0,E,-52.8,20.566038,626.0,D,-1.6
2,AB-004408,Cardenas Yedwab,3,7747.0,49,158.10,1071.0,0.0458,0.4297,5.0,...,E,-7.4,54.842767,357.0,C,0.7,64.465409,273.0,B,1.1
3,AH-00722,Garza Elijah,3,8070.0,52,155.19,1071.0,0.0486,0.4637,10.0,...,C,-0.1,55.660377,351.0,C,1.6,64.465409,273.0,B,1.1
4,AK-00157,Wyatt Pak,3,9489.0,62,153.05,1081.0,0.0574,0.4623,0.0,...,C,-0.4,77.735849,175.0,B,29.2,97.672956,8.0,A,2.1


In [12]:
# 2. 在 notebook 中准备可视化图形（不直接使用 .show()，避免 nbformat 依赖）

# 示例 1：总销售额 vs 购买频率，按聚类上色
fig_scatter = None
if "Cluster" in profiles_scored_df.columns:
    fig_scatter = px.scatter(
        profiles_scored_df,
        x="Total_Sales",
        y="Purchase_Frequency",
        color="Cluster",
        hover_data=["Customer_ID", "Customer_Name", "Avg_Order_Value"],
        title="客户聚类分布：总销售额 vs 购买频率",
    )
    print("已生成散点图对象 fig_scatter，可用于导出 HTML。")
else:
    print("当前数据中没有 'Cluster' 列，无法按聚类上色。请检查 k-means 处理是否已写回 CSV。")

# 示例 2：不同聚类的数量分布柱状图
fig_bar = None
if "Cluster" in profiles_scored_df.columns:
    cluster_counts = profiles_scored_df["Cluster"].value_counts().reset_index()
    cluster_counts.columns = ["Cluster", "Count"]

    fig_bar = px.bar(
        cluster_counts,
        x="Cluster",
        y="Count",
        text="Count",
        title="各聚类用户数量分布",
    )
    fig_bar.update_traces(textposition="outside")
    print("已生成柱状图对象 fig_bar，可用于导出 HTML。")

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
# 3. 导出为独立 HTML 网页（静态可交互图表）

from pathlib import Path

OUTPUT_HTML = "customer_dashboard.html"

if "Cluster" in profiles_scored_df.columns:
    # 简单示例：把上面的散点图导出为 HTML
    fig_export = px.scatter(
        profiles_scored_df,
        x="Total_Sales",
        y="Purchase_Frequency",
        color="Cluster",
        hover_data=["Customer_ID", "Customer_Name", "Avg_Order_Value"],
        title="客户聚类分布：总销售额 vs 购买频率",
    )

    fig_export.write_html(OUTPUT_HTML, include_plotlyjs="cdn", full_html=True)
    print(f"已导出交互式网页文件: {Path(OUTPUT_HTML).resolve()}")
else:
    print("当前数据中没有 'Cluster' 列，暂不导出 HTML。")

# 后续如果你想做真正的 Web 应用（如 Streamlit / Gradio），
# 可以在单独的 .py 文件中使用这里的逻辑封装成接口和页面。

In [ ]:
# 4. RFM-BC 各群体雷达图（各聚类核心指标均值，Min-Max 标准化到 0-100）

import plotly.graph_objects as go

# RFM-BC 对应特征（现有数据中可用的）
rfm_bc_cols = {
    "R-最近购买间隔": "Last_Purchase_Days_Ago",   # 逆向：天数越小越好
    "F-订单数": "Total_Orders",
    "M-销售额": "Total_Sales",
    "B-互动分": "Engagement_Score",
    "C-品类广度": "Unique_Products_Bought",
}
rfm_bc_cols = {k: v for k, v in rfm_bc_cols.items() if v in profiles_scored_df.columns}
if not rfm_bc_cols:
    rfm_bc_cols = {"销售额": "Total_Sales", "订单数": "Total_Orders", "客单价": "Avg_Order_Value",
                   "购买频率": "Purchase_Frequency", "互动分": "Engagement_Score"}
    rfm_bc_cols = {k: v for k, v in rfm_bc_cols.items() if v in profiles_scored_df.columns}

cluster_col = "Cluster_Label" if "Cluster_Label" in profiles_scored_df.columns else "Cluster"
if cluster_col not in profiles_scored_df.columns:
    cluster_col = "Cluster"

grp = profiles_scored_df.groupby(cluster_col)[list(rfm_bc_cols.values())].mean()
# 对 R（最近购买间隔）做逆向：值越大越“差”，用 max-val 转为越大越好
if "Last_Purchase_Days_Ago" in rfm_bc_cols.values():
    col_r = [k for k, v in rfm_bc_cols.items() if v == "Last_Purchase_Days_Ago"][0]
    grp["Last_Purchase_Days_Ago"] = grp["Last_Purchase_Days_Ago"].max() - grp["Last_Purchase_Days_Ago"]

# Min-Max 标准化到 0-100
for c in grp.columns:
    mi, mx = grp[c].min(), grp[c].max()
    grp[c] = (grp[c] - mi) / (mx - mi + 1e-9) * 100

categories = list(rfm_bc_cols.keys())
fig_radar = go.Figure()
for idx, row in grp.iterrows():
    vals = [float(row[c]) for c in rfm_bc_cols.values()]
    fig_radar.add_trace(go.Scatterpolar(r=vals + [vals[0]], theta=categories + [categories[0]],
                                        name=str(idx), fill="toself"))
fig_radar.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
                        title="各用户群体 RFM-BC 维度雷达图（0-100 标准化）")
# 导出 HTML 避免 .show() 依赖 nbformat
fig_radar.write_html("rfm_bc_radar.html", include_plotlyjs="cdn")
print("已导出 rfm_bc_radar.html")

In [ ]:
# 5. 各聚类 × 特征热力图（聚类均值，按行标准化）

import plotly.graph_objects as go

feature_cols = [
    "Total_Sales", "Total_Orders", "Avg_Order_Value", "Customer_Lifetime",
    "Purchase_Frequency", "Profit_Margin", "Engagement_Score",
]
feature_cols = [c for c in feature_cols if c in profiles_scored_df.columns]

cluster_col = "Cluster_Label" if "Cluster_Label" in profiles_scored_df.columns else "Cluster"
heat_df = profiles_scored_df.groupby(cluster_col)[feature_cols].mean()
# 按列 Min-Max 标准化到 0-100，便于比较各群体在各特征上的相对强弱
heat_norm = (heat_df - heat_df.min()) / (heat_df.max() - heat_df.min() + 1e-9) * 100

fig_heat = go.Figure(data=go.Heatmap(
    z=heat_norm.values,
    x=[c.replace("_", " ") for c in feature_cols],
    y=[str(i) for i in heat_norm.index],
    colorscale="RdYlGn",
    text=heat_df.round(1).values,
    texttemplate="%{text}",
    textfont={"size": 10},
))
fig_heat.update_layout(
    title="各用户群体关键特征热力图（行内 0-100 标准化，数字为原始均值）",
    xaxis_title="特征",
    yaxis_title="用户群体",
)
fig_heat.write_html("cluster_feature_heatmap.html", include_plotlyjs="cdn")
print("已导出 cluster_feature_heatmap.html")